In [1]:
import parsers
from common.utils.rune import get_client,get_watch_data,get_bilateral_watch_data
from common.utils.rclone import copy, list_remote
import subprocess
import os
import subprocess
import matlab.engine
import re
import shutil
from common.utils.time import unix_to_timestamps
from common.utils.ingest import storage_format_date
import numpy as np
import pandas as pd
from scipy import signal
import datetime
import matplotlib.pyplot as plt
import shutil


In [2]:
myRCSParser = parsers.RCSParser()
myRCSParser.full_parse()

START


usage: scp [-346BCpqrTv] [-c cipher] [-F ssh_config] [-i identity_file]
            [-J destination] [-l limit] [-o ssh_option] [-P port]
            [-S program] source ... target
usage: scp [-346BCpqrTv] [-c cipher] [-F ssh_config] [-i identity_file]
            [-J destination] [-l limit] [-o ssh_option] [-P port]
            [-S program] source ... target


Aggregate
Anonymize

failures =

  0x0 empty cell array

To csv

failures =

  0x0 empty cell array



In [ ]:
def get_new_session_names(ucsf_session_names):
    #Get dates folder names that has been processed and uploaded to wasabi
    current_dates = list_remote('rcs07/rcs_v2/')
    if '.DS_Store\n' in current_dates:
        current_dates.remove('.DS_Store\n')
        
    current_dates = current_dates[0:-3]
    current_dates = max([int(i[0:-2]) for i in current_dates])
    
    session_unix_times = [int(i[7:]) for i in ucsf_session_names]
    ucsf_session_dates = np.asarray([unix_to_timestamps(i).strftime('%Y%m%d') for i in session_unix_times])
    new_sessions_mask = np.asarray(ucsf_session_dates,dtype=int) > current_dates
    
    new_session_names = ucsf_session_names[new_sessions_mask]
    return new_session_names


def download_session_data(side, session_folder_names):
        session_paths = ["rbechto2@10.37.129.11:'/media/dropbox_hdd/Starr Lab Dropbox/RC+S Patient Un-Synced Data/RCS07 Un-Synced Data/SummitData/SummitContinuousBilateralStreaming/RCS07" + side + f"/{i}'" for i in session_folder_names]
        print( session_paths)
        p = subprocess.Popen(["scp", "-r" ,*session_paths, "./temp/combined_original/"])
        # TODO: change to p.communicate and get the output and error message
        p.wait(1800)

def upload_to_wasabi():
    #copy('./temp/combined_anonymized_json_csv/.', 'secret_sauce:/rcs07/rcs_v2/')
    return None

def clean_directory():
        shutil.rmtree('./temp/')
        os.mkdir('./temp')
        return None

In [ ]:
for i in ['L','R']:
    ucsf_all_files = subprocess.check_output(["ssh", "rbechto2@10.37.129.11","ls","/media/dropbox_hdd/Starr\ Lab\ Dropbox/RC+S\ Patient\ Un-Synced\ Data/RCS07\ Un-Synced\ Data/SummitData/SummitContinuousBilateralStreaming/RCS07"+i]).decode("utf-8")
    ucsf_session_names = [i for i in ucsf_all_files.split() if 'Session' in i]
    download_session_data(i,get_new_session_names(np.asarray(ucsf_session_names)))
    

In [ ]:
clean_directory()

In [ ]:

myclient = get_client()

time_range = [1657258984,1657259534]

wrist_params = {
'patient_id': 'rcs07',
'left_watch_id': '8QuY9OFb',
'right_watch_id': 'RElEtNme',
'time_range': time_range
}

rcs_params = {
'patient_id': 'rcs07',
'left_watch_id': 'NPC700419H',
'right_watch_id': 'NPC700403H',
'time_range': time_range
}

left_watch_params = {
    'patient_id': 'rcs07',
    'device_id': '8QuY9OFb',
    'start_time': time_range[0],
    'end_time': time_range[1]
}


# get_watch_data(myclient, right_watch_params , 'tremor')
# get_bilateral_watch_data(myclient, 'accel', **wrist_params)

In [ ]:
fft_size = 300
plt.figure()
plt.title('PSD of Right Apple Watch Acceleration')
a = np.linalg.norm(open_accel[1],axis=1)
f, Pxx_den = signal.periodogram(a, 50,nfft=fft_size)
plt.semilogy(f, Pxx_den)
plt.ylim([1e-9, 1e2])
# plt.xlabel('frequency [Hz]')
# plt.ylabel('PSD [V**2/Hz]')
# plt.show()

# plt.figure()
# plt.title('PSD of Right Apple Watch Acceleration (Adaptive Stim 2.4mA)')
a = np.linalg.norm(high_accel[1],axis=1)
f, Pxx_den = signal.periodogram(a, 50,nfft=fft_size)
plt.semilogy(f, Pxx_den)
# plt.ylim([1e-9, 1e2])
# plt.xlabel('frequency [Hz]')
# plt.ylabel('PSD [V**2/Hz]')
# plt.show()

# plt.figure()
# plt.title('PSD of Right Apple Watch Acceleration (Adaptive Stim 1.6mA)')
a = np.linalg.norm(low_accel[1],axis=1)
f, Pxx_den = signal.periodogram(a, 50,nfft=fft_size)
plt.semilogy(f, Pxx_den)
# plt.ylim([1e-9, 1e2])
# plt.xlabel('frequency [Hz]')
# plt.ylabel('PSD [V**2/Hz]')
# plt.show()
plt.legend(['Open','High','Low'])

In [ ]:
fig = plt.figure(figsize =(10, 7))
ax = fig.add_subplot(111)
data = np.squeeze(np.asarray([open_severity[0]['unknown'].iloc[1:],high_severity[0]['unknown'].iloc[1:],low_severity[0]['unknown'].iloc[1:]]))
ax.boxplot(data.T)
plt.title('Tremor Probability under 3 Different Stim Conditions')
plt.ylabel('Probability')
plt.ylim([0,1])
ax.set_xticklabels(['Open Stim (2mA)','Adaptaive Stim (2.4mA)','Adaptive Stim (1.6mA)'])

In [ ]:
np.asarray([open_severity[0]["slight"].iloc[1:],low_severity[0]["slight"].iloc[1:],high_severity[0]["slight"].iloc[1:]])

In [ ]:
open_severity[0]

In [ ]:
def get_timestamps(path_to_folder):
    neural_time_domain = pd.read_csv(path_to_folder + '/NeuralTimeDomain.csv')
    start = str(neural_time_domain["timestamp"].iloc[0])
    end = str(neural_time_domain["timestamp"].iloc[-1])
    time_range = [start[0:-4], end[0:-4]]
    return time_range

def get_side(folder_name):
    if 'left' in folder_name:
        return True
    if 'right' in folder_name:
        return False
    
def get_params(side,dual_sided_params):
    if side:
        return {
        'patient_id': dual_sided_params['patient_id'],
        'device_id': dual_sided_params['left_watch_id'],
        'start_time': dual_sided_params['time_range'][0],
        'end_time': dual_sided_params['time_range'][1]}
    else:
        return {
        'patient_id': dual_sided_params['patient_id'],
        'device_id': dual_sided_params['right_watch_id'],
        'start_time': dual_sided_params['time_range'][0],
        'end_time': dual_sided_params['time_range'][1]}
    
    
def timestamps_to_rune_data(timestamps,folder_name):


    wrist_params = {
    'patient_id': 'rcs07',
    'left_watch_id': '8QuY9OFb',
    'right_watch_id': 'RElEtNme',
    'time_range': timestamps
    }
    rcs_params = {
    'patient_id': 'rcs07',
    'left_watch_id': 'NPC700419H',
    'right_watch_id': 'NPC700403H',
    'time_range': timestamps
    }
        
    
    my_accel = get_watch_data(myclient, get_params(get_side(folder_name),wrist_params), 'accel').set_index('timestamp')
    my_rotation = get_watch_data(myclient, get_params(get_side(folder_name),wrist_params), 'rotation').set_index('timestamp')
    my_heart_rate = get_watch_data(myclient, get_params(get_side(folder_name),wrist_params), 'heart rate').set_index('timestamp')
    my_tremor = get_watch_data(myclient, get_params(get_side(folder_name),wrist_params), 'tremor').set_index('timestamp')
    my_tremor_severity = get_watch_data(myclient, get_params(get_side(folder_name),wrist_params), 'tremor severity').set_index('timestamp')
    my_dyskinesia = get_watch_data(myclient, get_params(get_side(folder_name),wrist_params), 'dyskinesia').set_index('timestamp')
    my_lfp = get_watch_data(myclient, get_params(get_side(folder_name),rcs_params), 'lfp').set_index('timestamp')
    my_band_power = get_watch_data(myclient, get_params(get_side(folder_name),rcs_params), 'band power').set_index('timestamp')
    
    return my_accel, my_rotation, my_heart_rate, my_tremor, my_tremor_severity, my_dyskinesia, my_lfp, my_band_power

def output_to_csv(path, date, folder, my_accel, my_rotation, my_heart_rate, my_tremor, my_tremor_severity, my_dyskinesia, my_lfp, my_band_power):

    if get_side(folder):
        full_path =  path + 'rune/' + date + '/rune_left_' + folder[-27:]
    else:
        full_path =  path + 'rune/' + date + '/rune_right_' + folder[-27:]
        
    # If folder doesn't exist, then create it.
    if not os.path.isdir(full_path):
        os.makedirs(full_path)

    
    my_accel.to_csv(full_path + '/accel.csv')
    my_rotation.to_csv(full_path + '/rotation.csv')
    my_heart_rate.to_csv(full_path + '/heart_rate.csv')
    my_tremor.to_csv(full_path + '/tremor.csv')
    my_tremor_severity.to_csv(full_path + '/tremor_severity.csv')
    my_dyskinesia.to_csv(full_path + '/dyskinesia.csv')
    my_lfp.to_csv(full_path + '/lfp.csv')
    my_band_power.to_csv(full_path + '/band_power.csv')


In [ ]:
folder_path = '/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data/rcs07/'
date_dir = os.listdir(folder_path + 'rcs/combined_anonymized_json_csv/')

if '.DS_Store' in date_dir:
    date_dir.remove('.DS_Store')

for i in date_dir:
    session_dir = os.listdir(folder_path + 'rcs/combined_anonymized_json_csv/'+i)
    if '.DS_Store' in session_dir:
        session_dir.remove('.DS_Store')
    for j in session_dir:
        my_timestamps = get_timestamps(folder_path + 'rcs/combined_anonymized_json_csv/'+i+'/'+j)    
        output_to_csv(folder_path,i,j,*timestamps_to_rune_data(my_timestamps,j))

## Dropbox Code

In [ ]:
#### DROPBOX_ACCESS_TOKEN = 'sl.BJ_794kQETs-AQ7jWJnc--vK2fCQ6RoPolVzvQ4vZEKdYfmYsr5WinUt3b-BCHlEEKjPuV59oGGaUA2ZPXI3JSyNy2F4MTe5bCVegs8z5WbXes0YZT7BF7H4tzX7K_HJPOrXo_Pc'
def dropbox_connect():
    """Create a connection to Dropbox."""

    try:
        dbx = dropbox.Dropbox(DROPBOX_ACCESS_TOKEN)
    except AuthError as e:
        print('Error connecting to Dropbox with access token: ' + str(e))
    return dbx

def dropbox_list_files():
    """Return a Pandas dataframe of files in a given Dropbox folder path in the Apps directory.
    """

    dbx = dropbox_connect()

    try:
        files = dbx.sharing_list_folders().entries
        files_list = []
        for file in files:
            if isinstance(file, dropbox.files.FileMetadata):
                metadata = {
                    'name': file.name,
                    'path_display': file.path_display,
                    'client_modified': file.client_modified,
                    'server_modified': file.server_modified
                }
                files_list.append(metadata)

        df = pd.DataFrame.from_records(files_list)
        return df.sort_values(by='server_modified', ascending=False)

    except Exception as e:
        print('Error getting list of files from Dropbox: ' + str(e))

dbx = dropbox.Dropbox(DROPBOX_ACCESS_TOKEN)
unscyned_shared_link = dbx.sharing_list_folders().entries[1].preview_url
dbx.files_list_folder('/SummitData/SummitContinuousBilateralStreaming/RCS07L',recursive=False, shared_link=dropbox.files.SharedLink(url=unscyned_shared_link)).entries[456]

In [ ]:
dbx.sharing_get_shared_link_file_to_file('/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data/rcs07/rcs/combined_original',unscyned_shared_link, path='/SummitData/SummitContinuousBilateralStreaming/RCS07L/Session1651792726456/.')

